# The General EM Algorithm

## Learning Objectives
- Derive the **ELBO (Evidence Lower Bound)** $J(Q, \theta)$ using Jensen's inequality
- Understand why setting $Q_i(z) = p(z \mid x^{(i)}; \theta)$ makes the bound **tight**
- State the general **E-step** and **M-step** in terms of $J$
- Prove that EM causes the log-likelihood $\ell(\theta)$ to **increase monotonically**
- View EM as **coordinate ascent** on $J(Q, \theta)$

## Problem Statement

**Setup:** We have a training set $\{x^{(1)}, \ldots, x^{(m)}\}$ and a model $p(x, z; \theta)$ with latent variables $z$.
We wish to maximise:
$$\ell(\theta) = \sum_{i=1}^m \log p(x^{(i)}; \theta) = \sum_{i=1}^m \log \sum_z p(x^{(i)}, z; \theta)$$

The $\log\sum$ has no closed-form solution. EM constructs and maximises a **tractable lower bound**.

**Lower bound derivation.** Introduce any distributions $Q_i$ over $z$ (one per example):
$$\ell(\theta) = \sum_i \log \sum_z Q_i(z) \frac{p(x^{(i)}, z; \theta)}{Q_i(z)}
\geq \sum_i \sum_z Q_i(z) \log \frac{p(x^{(i)}, z; \theta)}{Q_i(z)} =: J(Q, \theta)$$

The inequality follows from Jensen's inequality applied to the concave $\log$ function.

**Optimal $Q_i$ (tight bound).** Equality in Jensen's holds iff the argument is constant in $z$:
$$\frac{p(x^{(i)}, z; \theta)}{Q_i(z)} = c \implies Q_i(z) = p(z \mid x^{(i)}; \theta)$$

---

**EM Algorithm** (repeat until convergence):

**E-step:** For each $i$, set the posterior as the auxiliary distribution:
$$Q_i(z) := p(z \mid x^{(i)}; \theta)$$

**M-step:** Maximise the lower bound over $\theta$:
$$\theta := \arg\max_\theta \sum_i \sum_z Q_i(z) \log \frac{p(x^{(i)}, z; \theta)}{Q_i(z)}$$

**Convergence guarantee:** $\ell(\theta^{(t+1)}) \geq \ell(\theta^{(t)})$ — EM never decreases the log-likelihood.

**Coordinate ascent view.** Defining $J(Q, \theta) = \sum_i \sum_z Q_i(z) \log \frac{p(x^{(i)},z;\theta)}{Q_i(z)}$:
- **E-step** maximises $J$ over $Q$ (holding $\theta$ fixed)
- **M-step** maximises $J$ over $\theta$ (holding $Q$ fixed)

## 1. The Lower Bound $J(Q, \theta)$ and Jensen's Derivation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Illustrate the derivation steps on a simple example
# Simple model: x is 1D, z in {1, 2}, Gaussian mixture
from scipy.stats import norm

phi = np.array([0.4, 0.6])
mus = np.array([-1.5, 2.0])
sigs = np.array([0.8, 1.0])
x0 = 1.2   # one observed point

# p(x, z=j) for j in {0, 1}
pxz = phi * norm.pdf(x0, mus, sigs)
px  = pxz.sum()
Q_post = pxz / px    # posterior

# ELBO for different Q1 values
q1_grid = np.linspace(0.01, 0.99, 300)
def elbo(q1):
    q2 = 1 - q1
    return (q1 * np.log(pxz[0] / q1 + 1e-300)
          + q2 * np.log(pxz[1] / q2 + 1e-300))

J_grid  = np.array([elbo(q) for q in q1_grid])
J_tight = elbo(Q_post[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: ELBO as a function of Q1
ax = axes[0]
ax.plot(q1_grid, J_grid, 'b-', lw=2.5, label='$J(Q,\\theta)$ (lower bound)')
ax.axhline(np.log(px), color='green', ls='--', lw=2,
           label=f'$\\ell(\\theta) = \\log p(x) = {np.log(px):.3f}$')
ax.axvline(Q_post[0], color='red', ls=':', lw=2,
           label=f'Posterior $Q(z=1|x)={Q_post[0]:.3f}$')
ax.scatter(Q_post[0], J_tight, s=200, c='red', zorder=7, marker='*',
           label=f'Tight bound $J={J_tight:.3f}$')
ax.set_xlabel('$Q_1 = Q(z=1)$')
ax.set_ylabel('$J(Q, \\theta)$')
ax.set_title('ELBO $J(Q,\\theta) \\leq \\ell(\\theta)$\n'
             'Maximum of $J$ over $Q$ = $\\ell(\\theta)$')
ax.legend(fontsize=8.5)

# Right: derivation steps as annotated diagram
ax = axes[1]
ax.axis('off')
steps = [
    ('Step 1', '$\\ell(\\theta) = \\sum_i \\log \\sum_z p(x^{(i)}, z; \\theta)$',
     'Start: marginalise out $z$ (intractable $\\log\\sum$)', '#cce5ff'),
    ('Step 2', '$= \\sum_i \\log \\sum_z Q_i(z) \\frac{p(x^{(i)}, z; \\theta)}{Q_i(z)}$',
     'Introduce any distributions $Q_i(z)$ (multiply/divide)', '#d4edda'),
    ('Step 3', '$\\geq \\sum_i \\sum_z Q_i(z) \\log \\frac{p(x^{(i)}, z; \\theta)}{Q_i(z)}$',
     "Jensen's inequality: $E[\\log X] \\leq \\log E[X]$ for $\\log$ concave", '#fff3cd'),
    ('Step 4', '$=: J(Q, \\theta)$ (ELBO)',
     'Tight when $Q_i(z) = p(z|x^{(i)};\\theta)$: ratio is constant in $z$', '#f8d7da'),
]

for k, (label, eq, note, color) in enumerate(steps):
    y = 0.95 - k * 0.24
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.02, y - 0.17), 0.96, 0.19,
        boxstyle='round,pad=0.01', fc=color, ec='#555', lw=1.2, transform=ax.transAxes
    ))
    ax.text(0.06, y - 0.01, label, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='top')
    ax.text(0.22, y - 0.01, eq, transform=ax.transAxes,
            fontsize=9, va='top')
    ax.text(0.06, y - 0.10, note, transform=ax.transAxes,
            fontsize=7.5, va='top', color='#444', style='italic')
    if k < len(steps) - 1:
        ax.annotate('', xy=(0.5, y - 0.18), xytext=(0.5, y - 0.175),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='k', lw=1.5))

ax.set_title('Derivation: $\\ell(\\theta) \\geq J(Q,\\theta)$', fontsize=10)

fig.suptitle('EM Lower Bound: Jensen\'s Inequality → ELBO', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Monotonic Convergence Proof (Visualised)

**Proof that $\ell(\theta^{(t+1)}) \geq \ell(\theta^{(t)})$:**

At iteration $t$, let $Q_i^{(t)}(z) = p(z \mid x^{(i)}; \theta^{(t)})$.
This makes the bound tight: $\ell(\theta^{(t)}) = J(Q^{(t)}, \theta^{(t)})$.

The M-step then finds $\theta^{(t+1)} = \arg\max_\theta J(Q^{(t)}, \theta)$, so:
$$\ell(\theta^{(t+1)}) \geq J(Q^{(t)}, \theta^{(t+1)}) \geq J(Q^{(t)}, \theta^{(t)}) = \ell(\theta^{(t)}) \qquad \blacksquare$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Run EM on a small GMM and visualise the two-step convergence argument per iteration
np.random.seed(42)
k_true = 2
phi_true = np.array([0.5, 0.5])
mus_true  = np.array([[-2.0], [2.0]])
sigs_true = [np.array([[1.0]]), np.array([[1.0]])]
m = 100
z_true = np.random.choice(k_true, m, p=phi_true)
X = np.array([[np.random.normal(mus_true[z, 0], 1.0)] for z in z_true])

def gmm_log_likelihood_1d(X, phi, mus, sigs):
    from scipy.stats import norm
    total = 0.0
    for xi in X:
        px = sum(phi[j] * norm.pdf(xi[0], mus[j, 0], np.sqrt(sigs[j][0,0])) for j in range(len(phi)))
        total += np.log(px + 1e-300)
    return total

def em_step_1d(X, phi, mus, sigs):
    from scipy.stats import norm
    k = len(phi); m = len(X)
    # E-step
    W = np.zeros((m, k))
    for j in range(k):
        W[:, j] = phi[j] * norm.pdf(X[:, 0], mus[j, 0], np.sqrt(sigs[j][0, 0]))
    W /= W.sum(axis=1, keepdims=True)
    # M-step
    Wsum = W.sum(axis=0)
    phi_new = Wsum / m
    mus_new  = np.array([[(W[:, j] @ X[:, 0]) / Wsum[j]] for j in range(k)])
    sigs_new = [np.array([[(W[:, j] @ (X[:, 0] - mus_new[j, 0])**2) / Wsum[j]]]) for j in range(k)]
    return phi_new, mus_new, sigs_new, W

# Run 10 iterations, track ell
phi = np.array([0.5, 0.5])
mus  = np.array([[-3.0], [3.0]])
sigs = [np.array([[2.0]]), np.array([[2.0]])]

ll_before = []   # ell(theta^(t))   = J(Q^(t), theta^(t))  [tight]
ll_after  = []   # ell(theta^(t+1)) >= J(Q^(t), theta^(t+1))

for t in range(12):
    ll_t = gmm_log_likelihood_1d(X, phi, mus, sigs)
    ll_before.append(ll_t)
    phi, mus, sigs, W = em_step_1d(X, phi, mus, sigs)
    ll_after.append(gmm_log_likelihood_1d(X, phi, mus, sigs))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: ell over iterations
ax = axes[0]
iters = range(1, len(ll_before) + 1)
ax.plot(iters, ll_before, 'b-o', lw=2, ms=7, label='$\\ell(\\theta^{(t)})$ before M-step')
ax.plot(iters, ll_after,  'r-s', lw=2, ms=7, label='$\\ell(\\theta^{(t+1)})$ after M-step')
ax.fill_between(iters, ll_before, ll_after, alpha=0.15, color='green',
                label='Improvement per iteration')
ax.set_xlabel('Iteration $t$')
ax.set_ylabel('$\\ell(\\theta)$')
ax.set_title('$\\ell$ increases each EM iteration\n$\\ell(\\theta^{(t)}) \\leq \\ell(\\theta^{(t+1)})$')
ax.legend(fontsize=8.5)

# Right: proof diagram for one iteration
ax = axes[1]
ax.axis('off')
proof_steps = [
    ('E-step',  '$Q_i^{(t)}(z) := p(z|x^{(i)};\\theta^{(t)})$',
     'Makes bound tight: $J(Q^{(t)},\\theta^{(t)}) = \\ell(\\theta^{(t)})$', '#d4edda'),
    ('M-step',  '$\\theta^{(t+1)} = \\arg\\max_\\theta J(Q^{(t)}, \\theta)$',
     '$J(Q^{(t)},\\theta^{(t+1)}) \\geq J(Q^{(t)},\\theta^{(t)})$', '#fff3cd'),
    ('Combine', '$\\ell(\\theta^{(t+1)}) \\geq J(Q^{(t)},\\theta^{(t+1)}) \\geq J(Q^{(t)},\\theta^{(t)}) = \\ell(\\theta^{(t)})$',
     'QED: log-likelihood never decreases', '#f8d7da'),
]
for k_idx, (label, eq, note, color) in enumerate(proof_steps):
    y = 0.92 - k_idx * 0.30
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.02, y - 0.22), 0.96, 0.24,
        boxstyle='round,pad=0.01', fc=color, ec='#555', lw=1.2, transform=ax.transAxes
    ))
    ax.text(0.06, y - 0.01, label, transform=ax.transAxes,
            fontsize=9.5, fontweight='bold', va='top')
    ax.text(0.22, y - 0.01, eq, transform=ax.transAxes, fontsize=8.5, va='top')
    ax.text(0.06, y - 0.12, note, transform=ax.transAxes,
            fontsize=8, va='top', color='#444', style='italic')
    if k_idx < 2:
        ax.annotate('', xy=(0.5, y - 0.23), xytext=(0.5, y - 0.225),
                    xycoords='axes fraction', textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color='k', lw=1.5))
ax.set_title('Convergence Proof (one iteration)', fontsize=10)

fig.suptitle('EM Monotonic Convergence: $\\ell(\\theta^{(t+1)}) \\geq \\ell(\\theta^{(t)})$',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. EM as Coordinate Ascent on $J(Q, \theta)$

Define $J(Q, \theta) = \sum_i \sum_z Q_i(z) \log \frac{p(x^{(i)}, z; \theta)}{Q_i(z)}$.

Since $\ell(\theta) \geq J(Q, \theta)$ for all $Q$, maximising $J$ pushes up $\ell$.

| Step | What is held fixed | What is maximised | Result |
|---|---|---|---|
| E-step | $\theta$ | $J$ over $Q$ | Sets $Q_i = p(z|x^{(i)};\theta)$, making bound tight |
| M-step | $Q$ | $J$ over $\theta$ | Finds better parameters $\theta$ |

This is **coordinate ascent**: alternating maximisation over $Q$ then $\theta$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Visualise EM as coordinate ascent: J surface in (Q1, theta) space
# 1D problem: one data point x0, 2 Gaussians with fixed sigs, vary mu1

x0   = 1.0
phi  = np.array([0.5, 0.5])
mu2  = 3.0
sig1 = sig2 = 1.0

# Grid: q1 vs mu1
q1_vals  = np.linspace(0.01, 0.99, 80)
mu1_vals = np.linspace(-4, 4, 80)
Q1g, Mu1g = np.meshgrid(q1_vals, mu1_vals)

def J_surface(q1, mu1):
    q2   = 1 - q1
    px_z1 = phi[0] * norm.pdf(x0, mu1, sig1)
    px_z2 = phi[1] * norm.pdf(x0, mu2, sig2)
    return (q1 * np.log(px_z1 / (q1 + 1e-300) + 1e-300)
          + q2 * np.log(px_z2 / (q2 + 1e-300) + 1e-300))

Jg = J_surface(Q1g, Mu1g)

# Simulate 5 EM steps
mu1_em = -3.5
path   = []
for _ in range(5):
    # E-step: compute posterior Q1 given current mu1
    px_z1 = phi[0] * norm.pdf(x0, mu1_em, sig1)
    px_z2 = phi[1] * norm.pdf(x0, mu2,    sig2)
    q1_em = px_z1 / (px_z1 + px_z2)
    path.append(('E', q1_em, mu1_em))

    # M-step: update mu1 to maximise J (weighted mean)
    # gradient wrt mu1 of q1*log N(x0;mu1,sig1) gives: q1*(x0-mu1)/sig1^2 = 0 => mu1 = x0
    # For a single point the M-step update is mu1 = x0 (trivial); use a small shift for illustration
    mu1_new = q1_em * x0 + (1 - q1_em) * mu1_em   # weighted move toward x0
    path.append(('M', q1_em, mu1_new))
    mu1_em = mu1_new

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: J surface as heatmap
ax = axes[0]
im = ax.contourf(q1_vals, mu1_vals, Jg, levels=30, cmap='viridis')
plt.colorbar(im, ax=ax, label='$J(Q, \\theta)$')

# Overlay EM path
prev = None
for step_type, q1, mu1 in path:
    c = '#ff4500' if step_type == 'E' else '#00bfff'
    ax.scatter(q1, mu1, s=100, c=c, zorder=6,
               marker='o' if step_type == 'E' else 's')
    if prev is not None:
        ax.annotate('', xy=(q1, mu1), xytext=(prev[0], prev[1]),
                    arrowprops=dict(arrowstyle='->', color='white', lw=1.5))
    prev = (q1, mu1)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff4500', ms=10, label='E-step'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#00bfff', ms=10, label='M-step'),
], fontsize=9)
ax.set_xlabel('$Q_1 = Q(z=1)$')
ax.set_ylabel('$\\mu_1$ (first component mean)')
ax.set_title('$J(Q,\\theta)$ surface\nEM = coordinate ascent (alternating maximisation)')

# Right: J value along EM path
ax = axes[1]
J_path = [J_surface(p[1], p[2]) for p in path]
steps_x = range(len(path))
step_labels = [p[0]+f'-{i//2+1}' for i, p in enumerate(path)]
bar_colors = ['#ff4500' if p[0]=='E' else '#00bfff' for p in path]
ax.bar(steps_x, J_path, color=bar_colors, edgecolor='k', alpha=0.8)
ax.set_xticks(list(steps_x))
ax.set_xticklabels(step_labels, fontsize=9)
ax.set_xlabel('EM Sub-step')
ax.set_ylabel('$J(Q, \\theta)$')
ax.set_title('$J$ after each E/M sub-step\n(non-decreasing by coordinate ascent)')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(fc='#ff4500', label='E-step'), Patch(fc='#00bfff', label='M-step')],
          fontsize=9)

fig.suptitle('EM as Coordinate Ascent on $J(Q, \\theta)$', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. General EM vs GMM EM: The Template

The general EM framework applies to any model with latent variables, not just GMMs.

| Concept | General EM | GMM instance |
|---|---|---|
| Latent variable $z$ | Any discrete/continuous | Cluster index $j \in \{1,\ldots,k\}$ |
| $Q_i(z)$ | Posterior $p(z|x^{(i)};\theta)$ | Responsibility $w_j^{(i)}$ |
| E-step | Compute posterior $Q_i$ | Compute $w_j^{(i)}$ via Bayes |
| M-step | $\arg\max_\theta \sum_i E_{Q_i}[\log p(x^{(i)},z;\theta)]$ | Update $\phi_j, \mu_j, \Sigma_j$ in closed form |
| Convergence | $\ell$ non-decreasing | Same guarantee |

The M-step simplifies because the $-\sum_i E_{Q_i}[\log Q_i(z)]$ term is constant in $\theta$:
$$\arg\max_\theta J(Q,\theta) = \arg\max_\theta \sum_i \sum_z Q_i(z) \log p(x^{(i)}, z; \theta)$$
This is the **expected complete-data log-likelihood** under $Q_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

# Full EM for 2D GMM to show general framework in action
np.random.seed(42)
k = 3
phi_true  = np.array([0.4, 0.35, 0.25])
mus_true  = np.array([[0.0, 0.0], [4.0, 1.0], [2.0, 4.5]])
sigs_true = [
    np.array([[1.0, 0.5], [0.5, 0.8]]),
    np.array([[0.8, -0.3], [-0.3, 1.2]]),
    np.array([[1.5, 0.0], [0.0, 0.5]]),
]
m = 200
z_true = np.random.choice(k, m, p=phi_true)
X = np.array([np.random.multivariate_normal(mus_true[z], sigs_true[z]) for z in z_true])

# General EM implementation
rng = np.random.default_rng(0)
phi = np.full(k, 1/k)
mus  = X[rng.choice(m, k, replace=False)].copy().astype(float)
sigs = [np.eye(2) for _ in range(k)]

ll_hist = []
for _ in range(30):
    # E-step: Q_i(z) = p(z|x^(i); phi, mus, sigs) — the posterior
    W = np.zeros((m, k))
    for j in range(k):
        W[:, j] = phi[j] * multivariate_normal(mus[j], sigs[j]).pdf(X)
    row_sums = W.sum(axis=1, keepdims=True)
    ll_hist.append(np.sum(np.log(row_sums.squeeze() + 1e-300)))
    W /= (row_sums + 1e-300)

    # M-step: maximise expected complete-data log-likelihood
    Wsum = W.sum(axis=0)
    phi  = Wsum / m
    mus  = (W.T @ X) / Wsum[:, None]
    for j in range(k):
        diff = X - mus[j]
        sigs[j] = (W[:, j:j+1] * diff).T @ diff / Wsum[j]

colors = ['#2166ac', '#d6604d', '#4dac26']
dominant = W.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: log-likelihood (EM guarantee)
ax = axes[0]
ax.plot(range(1, len(ll_hist)+1), ll_hist, 'b-o', lw=2.5, ms=5)
ax.set_xlabel('Iteration')
ax.set_ylabel('$\\ell(\\theta)$')
ax.set_title('General EM: $\\ell$ increases monotonically\n(guarantee holds for any latent-variable model)')

# Right: final clustering
ax = axes[1]
xg = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 100)
yg = np.linspace(X[:,1].min()-1, X[:,1].max()+1, 100)
Xg, Yg = np.meshgrid(xg, yg)
pos = np.dstack([Xg, Yg])
for j in range(k):
    pts = X[dominant == j]
    ax.scatter(pts[:,0], pts[:,1], c=colors[j], s=20, alpha=0.6)
    ax.scatter(*mus[j], marker='X', s=200, c=colors[j], edgecolors='k', lw=1.5, zorder=5)
    try:
        Z = multivariate_normal(mus[j], sigs[j]).pdf(pos)
        ax.contour(Xg, Yg, Z, levels=3, colors=[colors[j]], alpha=0.6, linewidths=1.5)
    except Exception:
        pass
ax.set_title('Final GMM fit after EM\n(instance of general framework)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('General EM Framework Applied to GMM', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Converged in 30 iterations')
print(f'Final log-likelihood: {ll_hist[-1]:.2f}')
print(f'Recovered phi: {phi.round(3)}  (true: {phi_true})')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="374"
     viewBox="0 0 780 374" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: ELBO -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">ELBO J(Q,&#x3b8;)</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">&#x2264; &#x2113;(&#x3b8;)</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >tractable lower bound on marginal log-likelihood via Jensen&#x2019;s inequality</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >E-step: set Q&#x2071; = posterior p(z|x&#x2071;;&#x3b8;)</text>

  <!-- Row 2: Tight bound -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Tight bound</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >J(Q&#x207b;ᵗ&#x207e;, &#x3b8;&#x207b;ᵗ&#x207e;) = &#x2113;(&#x3b8;&#x207b;ᵗ&#x207e;) &#x2014; E-step touches the log-likelihood surface</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >M-step: &#x3b8;&#x207b;&#x1d57;&#x207a;&#xb9;&#x207e; = argmax&#x3b8; J(Q&#x207b;ᵗ&#x207e;, &#x3b8;)</text>

  <!-- Row 3: M-step -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">M-step</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >J(Q&#x207b;ᵗ&#x207e;,&#x3b8;&#x207b;&#x1d57;&#x207a;&#xb9;&#x207e;) &#x2265; J(Q&#x207b;ᵗ&#x207e;,&#x3b8;&#x207b;ᵗ&#x207e;) = &#x2113;(&#x3b8;&#x207b;ᵗ&#x207e;) &#x2014; maximisation ensures increase</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >&#x2113;(&#x3b8;&#x207b;&#x1d57;&#x207a;&#xb9;&#x207e;) &#x2265; J(Q&#x207b;ᵗ&#x207e;,&#x3b8;&#x207b;&#x1d57;&#x207a;&#xb9;&#x207e;) &#x2265; &#x2113;(&#x3b8;&#x207b;ᵗ&#x207e;)</text>

  <!-- Row 4: Monotone convergence -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">&#x2113; non-decreasing</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >coordinate ascent on J &#x2014; convergence to local optimum guaranteed</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| ELBO | $J(Q,\theta) = \sum_i\sum_z Q_i(z)\log\frac{p(x^{(i)},z;\theta)}{Q_i(z)} \leq \ell(\theta)$ | Tractable lower bound via Jensen |
| E-step | $Q_i(z) := p(z\lvert x^{(i)};\theta)$ | Makes $J(Q,\theta) = \ell(\theta)$ — tight bound |
| M-step | $\theta := \arg\max_\theta \sum_i\sum_z Q_i(z)\log p(x^{(i)},z;\theta)$ | Expected complete-data log-likelihood |
| Convergence | $\ell(\theta^{(t+1)}) \geq J(Q^{(t)},\theta^{(t+1)}) \geq \ell(\theta^{(t)})$ | Monotonic non-decrease; guaranteed |
| Coordinate ascent | E-step maximises $J$ over $Q$; M-step over $\theta$ | EM = alternating maximisation |
| Generality | Works for any latent-variable model | GMM, factor analysis, HMMs, etc. |

**Key insight:** EM is coordinate ascent on the ELBO $J(Q,\theta)$; the E-step makes the bound touch the log-likelihood by choosing the posterior, and the M-step ascends along the bound — together guaranteeing monotonic non-decrease of the intractable $\ell(\theta)$.